# CSCI E-89, Lecture 11 — Homework
## RAG over four documents on different subjects (politics, sport, cooking, travel)

Starting point: `Lecture11_RAG_with_OpenAI.ipynb`. Changes:

1. Documents live in `data/` with **two subdirectories**, two PDFs each:
   ```
   data/current_affairs/politics_city_council_reform.pdf
   data/current_affairs/sport_harbor_marathon.pdf
   data/lifestyle/cooking_saffron_paella.pdf
   data/lifestyle/travel_azores_itinerary.pdf
   ```
   (generated by `make_rag_pdfs.py`; each file is 2 pages with distinctive facts).
2. All four PDFs are indexed into one Chroma vector store.
3. Four questions, each targeting one subject, are submitted to the LLM.
4. We inspect the sources visually and then **extract the source programmatically** in three ways.

In [ ]:
#!pip install -q langchain_openai langchain_chroma langchain_community pypdf python-dotenv reportlab

In [ ]:
import os, shutil
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv
try:                                    # same loader as in the lecture notebook
    from langchain_classic.document_loaders import PyPDFLoader
except ImportError:
    from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv()   # reads OPENAI_API_KEY from .env

#### Create the data directory (only needed once)

In [ ]:
if not Path("data").exists():
    !python make_rag_pdfs.py
for p in sorted(Path("data").rglob("*.pdf")):
    print(p)

#### Load all PDFs from both subdirectories
We add two extra metadata fields to every page: `subject` (taken from the file name prefix)
and `folder` (the subdirectory). They travel with every chunk into the vector store.

In [ ]:
DATA_DIR = Path("data")

documents = []
for pdf_path in sorted(DATA_DIR.rglob("*.pdf")):
    pages = PyPDFLoader(str(pdf_path)).load()
    for page in pages:
        page.metadata["subject"] = pdf_path.stem.split("_")[0]   # politics / sport / cooking / travel
        page.metadata["folder"] = pdf_path.parent.name           # current_affairs / lifestyle
        page.metadata["file"] = pdf_path.name
    documents.extend(pages)

print(f"Loaded {len(documents)} pages from {len({d.metadata['file'] for d in documents})} files")

#### Split into chunks

In [ ]:
# The PDFs are small, so use smaller chunks than in the lecture to get several chunks per file
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

print(f"{len(chunks)} chunks")
print(Counter(c.metadata["subject"] for c in chunks))

#### Embed and store in a fresh ChromaDB

In [ ]:
PERSIST_DIR = "./vectorstore_4subjects"
shutil.rmtree(PERSIST_DIR, ignore_errors=True)   # avoid duplicate chunks when re-running

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings,
                                    persist_directory=PERSIST_DIR)
print(f"Successfully indexed {len(chunks)} chunks")

### Build the RAG chatbot

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(model="gpt-6.1-sol", temperature=0)   # temperature 0 -> reproducible answers

PROMPT = '''
Answer the question using ONLY the information provided in the context.
If the answer is not available, say you don't know.
Context:
{context}
Question:
{question}
Answer:
'''

def ask_question(question, k=4):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    response = llm.invoke(PROMPT.format(context=context, question=question))

    print("QUESTION:", question)
    print("\nANSWER:")
    print(response.content)
    print("\nSOURCES:")
    for doc in docs:
        print(f"  {doc.metadata['folder']}/{doc.metadata['file']}  (page {doc.metadata['page']})")
    return response.content, docs

#### Four questions, one per subject
Each question asks for a fact that appears in only one document.

In [ ]:
questions = {
    "politics": "What percentage of the vote did Proposition 7 receive, and how many council members will there be?",
    "sport":    "Who won the men's race in the Harbor City Marathon and what was the winning time?",
    "cooking":  "How long should the paella rest after cooking, and what is the crispy bottom layer called?",
    "travel":   "How high is Mount Pico and how long does the climb take?",
}

results = {}
for subject, q in questions.items():
    print("=" * 90)
    results[subject] = ask_question(q)

#### Visual examination

Look at the `SOURCES:` block under each answer. If RAG works, the retrieved chunks for each question
come from the matching file (the politics question should list `politics_city_council_reform.pdf`, and so on).
Because the four subjects are very different, the embeddings separate them well, and usually
all, or nearly all, of the k=4 chunks come from the correct file. When the top chunks fill only part of k,
the remaining slots are taken by the "least unrelated" chunks from other files. Those chunks appear in the
source list but contribute nothing to the answer. So **the raw source list is not by itself a reliable
attribution**. We need to decide which source was actually used.

### Extracting the source programmatically

**Method 1: majority vote over retrieved chunks, weighted by relevance score.**
`similarity_search_with_relevance_scores` returns a score in [0, 1] for each chunk.
We sum the scores per file and pick the file with the highest total.

In [ ]:
def source_by_retrieval(question, k=4):
    hits = vectorstore.similarity_search_with_relevance_scores(question, k=k)
    score_per_file = Counter()
    for doc, score in hits:
        score_per_file[doc.metadata["file"]] += score
    best_file, _ = score_per_file.most_common(1)[0]
    return best_file, hits

for subject, q in questions.items():
    best, hits = source_by_retrieval(q)
    print(f"\n[{subject}] -> predicted source: {best}")
    for doc, score in hits:
        print(f"    {score:.3f}  {doc.metadata['file']}  p.{doc.metadata['page']}")

**Method 2: ask the LLM to cite the source it used.**
Each chunk in the context is labelled with its file name, and the LLM returns a structured answer
(`with_structured_output` + Pydantic) that includes the file(s) it relied on.
This reports which chunks were *used*, not just which ones were *retrieved*.

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class CitedAnswer(BaseModel):
    answer: str = Field(description="Answer to the question, based only on the context")
    sources: List[str] = Field(description="File names of the context passages actually used for the answer")

cited_llm = llm.with_structured_output(CitedAnswer)

CITED_PROMPT = '''
Answer the question using ONLY the context below. Each passage is labelled with its source file.
If the answer is not available, say you don't know and return an empty list of sources.
In `sources`, list ONLY the files whose passages you actually used.

Context:
{context}

Question: {question}
'''

def ask_with_citations(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(f"[source: {d.metadata['file']}]\n{d.page_content}" for d in docs)
    return cited_llm.invoke(CITED_PROMPT.format(context=context, question=question)), docs

cited = {}
for subject, q in questions.items():
    out, docs = ask_with_citations(q)
    cited[subject] = out
    print(f"\n[{subject}] {out.answer}\n    cited: {out.sources}")

**Method 3: automatic check against the expected subject.**
Because we wrote the questions, we know the correct file. We compare both methods with it.

In [ ]:
import pandas as pd

rows = []
for subject, q in questions.items():
    retrieved = [d.metadata["subject"] for d in retriever.invoke(q)]
    by_retrieval, _ = source_by_retrieval(q)
    by_llm = cited[subject].sources
    rows.append({
        "expected": subject,
        "retrieved chunk subjects": retrieved,
        "precision@4": sum(s == subject for s in retrieved) / len(retrieved),
        "method 1 (score vote)": by_retrieval,
        "method 1 correct": by_retrieval.startswith(subject),
        "method 2 (LLM cited)": by_llm,
        "method 2 correct": len(by_llm) > 0 and all(f.startswith(subject) for f in by_llm),
    })

report = pd.DataFrame(rows)
report

#### Bonus: an off-topic question
None of the documents covers this question. The retriever still returns 4 chunks, because it always does.
The LLM should answer *I don't know* and cite no sources, and the relevance scores should be low.
This is why the citation and the score are more trustworthy than the plain source list.

In [ ]:
q = "Who won the 2022 FIFA World Cup?"
out, _ = ask_with_citations(q)
print(out)
_, hits = source_by_retrieval(q)
print([(round(s, 3), d.metadata["file"]) for d, s in hits])

### Conclusions
* **Visual check.** The `SOURCES` metadata printed under each answer shows which PDF (and page) each
  retrieved chunk came from. For four very different subjects, the top-ranked chunks come from the correct file.
* **Programmatic extraction.** Yes, it is possible. Every chunk carries `metadata["source"]`, and we added
  `subject`, `folder` and `file`.
  - *Method 1*: aggregate relevance scores per file. This is cheap and needs no extra LLM call.
    It only says which file was most similar.
  - *Method 2*: label the passages and ask the LLM to return the files it used (structured output).
    This reports actual use, which matters when the retriever pads k with irrelevant chunks
    or when nothing relevant exists (the off-topic question).
  - *Method 3*: comparing with the known subject gives an automatic accuracy check (`report` table).